=============================================================================
NOTEBOOK: eng_models.ipynb
Language: English
Models  : TTS → Kokoro-82M (StyleTTS2) | ASR → whisper-large-v3-turbo
GPU     : cuda:1  (Quadro RTX 6000 — 24 GB)
Port    : 8891
=============================================================================

CRITICAL system dependency — run ONCE before starting this notebook:
sudo apt-get install -y espeak-ng

Then install Python packages (Cell 1 below).

In [ ]:
!sudo apt-get install -y espeak-ng
!pip install kokoro>=0.9.2 soundfile misaki transformers>=4.40.0 \
             accelerate scipy flask flask-cors jupyter-server-proxy --quiet

In [ ]:
import sys
sys.path.insert(0, "/home/yniyonshuti/API")

import torch
import numpy as np
import base64, io, threading

from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from kokoro import KPipeline
from transformers import (
    AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline as hf_pipeline,
)

from utils.common import (
    setup_logger, Timer,
    waveform_to_wav_bytes, decode_audio_input,
    tts_response, asr_response, error_response,
)

BASE        = "/home/yniyonshuti/API"
ENG_TTS_DIR = f"{BASE}/Kokoro-82M"        # contains kokoro-v1_0.pth + voices/
ENG_ASR_DIR = f"{BASE}/whisper-large-v3-turbo"

GPU    = "cuda:1"
PORT   = 8891
logger = setup_logger("eng_api")

logger.info(f"Device: {GPU} — {torch.cuda.get_device_name(1)}")

# Available Kokoro voices (54 total — expose common ones)
KOKORO_VOICES = {
    "af_heart"   : "Female — warm American",
    "af_alloy"   : "Female — clear American",
    "af_nova"    : "Female — bright American",
    "am_echo"    : "Male — deep American",
    "am_onyx"    : "Male — rich American",
    "bf_emma"    : "Female — British",
    "bm_george"  : "Male — British",
}
DEFAULT_VOICE = "af_heart"

In [ ]:
logger.info("Loading EN TTS (Kokoro-82M) ...")
# KPipeline expects the model weights path; lang_code 'a' = American English
kokoro_pipe = KPipeline(lang_code="a", repo_id=ENG_TTS_DIR)
ENG_TTS_SR  = 24000   # Kokoro always outputs 24 kHz
logger.info("EN TTS (Kokoro) loaded. SR: 24000 Hz")

In [ ]:
logger.info("Loading EN ASR (whisper-large-v3-turbo) ...")
eng_asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    ENG_ASR_DIR,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    use_safetensors=True,
)
eng_asr_model.to(GPU)

eng_asr_processor = AutoProcessor.from_pretrained(ENG_ASR_DIR)

eng_asr_pipe = hf_pipeline(
    "automatic-speech-recognition",
    model=eng_asr_model,
    tokenizer=eng_asr_processor.tokenizer,
    feature_extractor=eng_asr_processor.feature_extractor,
    torch_dtype=torch.float16,
    device=GPU,
)
logger.info("EN ASR loaded.")

In [ ]:
def run_eng_tts(text: str, voice: str = DEFAULT_VOICE) -> dict:
    """
    Synthesise English speech using Kokoro-82M.
    voice: one of KOKORO_VOICES keys (default 'af_heart')
    Handles long text automatically via generator chunking.
    """
    if not text.strip():
        raise ValueError("text must not be empty")
    if voice not in KOKORO_VOICES:
        voice = DEFAULT_VOICE

    chunks = []
    with Timer() as t:
        generator = kokoro_pipe(text, voice=voice)
        for _, _, audio_chunk in generator:
            chunks.append(audio_chunk)

    waveform = np.concatenate(chunks).astype(np.float32)
    return tts_response(waveform, ENG_TTS_SR, text, "eng", f"Kokoro-82M/{voice}", t.elapsed_ms)


def run_eng_asr(audio_array: np.ndarray, src_sr: int,
                timestamps: bool = False, task: str = "transcribe") -> dict:
    if src_sr != 16000:
        import librosa
        audio_array = librosa.resample(audio_array, orig_sr=src_sr, target_sr=16000)
        src_sr = 16000

    audio_input = {"array": audio_array, "sampling_rate": src_sr}
    generate_kw = {"language": "english", "task": task}
    with Timer() as t:
        result = eng_asr_pipe(audio_input, return_timestamps=timestamps,
                              generate_kwargs=generate_kw)
    duration_s = round(len(audio_array) / src_sr, 3)
    return asr_response(result["text"], result.get("chunks", []),
                        "eng", "whisper-large-v3-turbo", t.elapsed_ms, duration_s)

In [ ]:
res = run_eng_tts("Hello! The English TTS API is running successfully.", voice="af_heart")
logger.info(f"EN TTS smoke test OK — {res['audio_duration_seconds']}s in {res['inference_time_ms']}ms")

In [ ]:
app = Flask("eng_api")
CORS(app)


@app.route("/health", methods=["GET"])
def health():
    mem = torch.cuda.memory_allocated(1) / 1024**3
    return jsonify({
        "status": "ok",
        "service": "english",
        "port": PORT,
        "device": GPU,
        "gpu_name": torch.cuda.get_device_name(1),
        "vram_used_gb": round(mem, 2),
        "models": {
            "tts": "Kokoro-82M",
            "asr": "whisper-large-v3-turbo (language=english)",
        },
        "available_voices": KOKORO_VOICES,
    })


@app.route("/tts", methods=["POST"])
def tts():
    """
    POST /tts
    {
      "text": "Hello world, this is a test.",
      "voice": "af_heart",       // optional — see /health for all voices
      "return_binary": false      // optional — return raw WAV instead of JSON
    }
    """
    body = request.get_json(force=True, silent=True) or {}
    text = (body.get("text") or "").strip()
    if not text:
        return jsonify(*error_response("'text' field is required"))

    voice         = body.get("voice", DEFAULT_VOICE)
    return_binary = bool(body.get("return_binary", False))

    try:
        result = run_eng_tts(text, voice)
        logger.info(f"TTS OK | voice={voice} | {result['audio_duration_seconds']}s "
                    f"in {result['inference_time_ms']}ms")
        if return_binary:
            wav_bytes = base64.b64decode(result["audio_base64"])
            return send_file(io.BytesIO(wav_bytes), mimetype="audio/wav",
                             as_attachment=True, download_name="eng_output.wav")
        return jsonify(result)
    except Exception as e:
        logger.error(f"TTS error: {e}")
        return jsonify(*error_response(str(e), 500))


@app.route("/tts/voices", methods=["GET"])
def voices():
    """GET /tts/voices — List all available Kokoro voices."""
    return jsonify({"voices": KOKORO_VOICES, "default": DEFAULT_VOICE})


@app.route("/asr", methods=["POST"])
def asr():
    """
    POST /asr
    JSON: { "audio_base64": "...", "timestamps": false, "task": "transcribe" }
    Multipart: file=<audio>, timestamps=true, task=transcribe
    task: "transcribe" (default) | "translate" → English output regardless of input lang
    """
    timestamps = False
    task = "transcribe"
    try:
        if request.content_type and "multipart" in request.content_type:
            if "file" not in request.files:
                return jsonify(*error_response("No 'file' field in request"))
            audio_bytes = request.files["file"].read()
            timestamps  = request.form.get("timestamps", "false").lower() == "true"
            task        = request.form.get("task", "transcribe")
            import soundfile as sf
            data, src_sr = sf.read(io.BytesIO(audio_bytes), dtype="float32")
            audio_array  = data.mean(axis=1) if data.ndim > 1 else data
        else:
            body = request.get_json(force=True, silent=True) or {}
            if not body.get("audio_base64"):
                return jsonify(*error_response("'audio_base64' is required"))
            timestamps  = bool(body.get("timestamps", False))
            task        = body.get("task", "transcribe")
            audio_array, src_sr = decode_audio_input(body["audio_base64"])

        result = run_eng_asr(audio_array, src_sr, timestamps, task)
        logger.info(f"ASR OK | '{result['transcript'][:60]}' in {result['inference_time_ms']}ms")
        return jsonify(result)
    except Exception as e:
        logger.error(f"ASR error: {e}")
        return jsonify(*error_response(str(e), 500))

In [ ]:
def _run():
    app.run(host="0.0.0.0", port=PORT, debug=False, use_reloader=False)

threading.Thread(target=_run, daemon=True).start()
logger.info(f"English API live on http://0.0.0.0:{PORT}")
print(f"\n✅ English API running — port {PORT}")
print(f"   GET  /health")
print(f"   GET  /tts/voices")
print(f"   POST /tts    body: {{\"text\": \"...\", \"voice\": \"af_heart\"}}")
print(f"   POST /asr    body: {{\"audio_base64\": \"...\"}}")